# **1. Text Preprocessing and TF-IDF Functions**

In [2]:
import re
import math
from collections import Counter

def preprocess(sentence):
    # 1. Lowercase
    sentence = sentence.lower()

    # 2. Handle URLs (Regex for http/https/www)
    sentence = re.sub(r'http\S+|www\.\S+', 'URL', sentence)

    # 3. Handle Numbers (Regex for digits)
    sentence = re.sub(r'\d+', 'NUMBER', sentence)

    # 4. Handle Punctuation (Regex for standard punctuation)
    # We replace punctuation with space+PUNCT+space to treat it as a token
    sentence = re.sub(r'[^\w\s]', ' PUNCT ', sentence)

    # 5. Tokenize (Split by whitespace and remove empty strings)
    tokens = sentence.split()

    return tokens

def compute_tf_with_normalization(sentence, vocab, smoothing=False):
    # TF = log(1 + (count(t, d) / total_words_in_d))
    # Note: The prompt asks for normalization based on total words.

    term_counts = Counter(sentence)
    total_words = len(sentence)
    tf_vector = {}

    for term in vocab:
        count = term_counts.get(term, 0)

        if smoothing:
            # Simple smoothing: add 1 to count to avoid zero, or similar logic
            # Interpreting "smoothing to handle unseen" as Laplace for TF is uncommon,
            # but we can ensure non-zero TF for vocab terms if requested.
            # Here we stick to standard log normalization.
            count += 1

        if total_words > 0:
            normalized_freq = count / total_words
        else:
            normalized_freq = 0

        # Apply Log score as requested in main
        # Using log(1 + x) to handle 0 frequencies naturally
        tf_score = math.log(1 + normalized_freq)
        tf_vector[term] = tf_score

    return tf_vector

def compute_idf(sentence, sentences, vocab, smoothing=False):
    # IDF = log(Total_Docs / (Doc_Freq + smoothing_factor))

    N = len(sentences)
    idf_vector = {}

    for term in vocab:
        # Count documents containing the term
        doc_freq = sum(1 for s in sentences if term in s)

        denominator = doc_freq
        if smoothing:
            denominator += 1  # Add 1 to avoid division by zero for unseen words

        # Apply log score
        # Using 1 + log to ensure IDF is positive and standard
        if denominator > 0:
            idf_score = math.log(N / denominator)
        else:
            idf_score = 0.0 # Should not happen if term is in vocab derived from sentences

        idf_vector[term] = idf_score

    return idf_vector

def compute_tf_idf_scores(sentences):
    # 1. Preprocess all sentences
    preprocessed_sentences = [preprocess(s) for s in sentences]

    # 2. Build Vocabulary (set of all unique tokens)
    vocab = sorted(list(set(token for s in preprocessed_sentences for token in s)))

    # 3. Compute IDFs (Computed once for the corpus)
    # We pass the first sentence just to satisfy signature, but IDF uses all 'sentences'
    idf_scores = compute_idf(None, preprocessed_sentences, vocab, smoothing=True)

    results = []

    # 4. Compute TF-IDF for each sentence
    for i, sent_tokens in enumerate(preprocessed_sentences):
        tf_scores = compute_tf_with_normalization(sent_tokens, vocab, smoothing=False)

        tf_idf_vector = {}
        for term in vocab:
            tf_idf_vector[term] = tf_scores[term] * idf_scores[term]

        results.append({
            "original": sentences[i],
            "tokens": sent_tokens,
            "tf_idf": tf_idf_vector
        })

    return results

def main():
    # Example Data
    raw_data = [
        "Order 3 items, get 1 free!",
        "Check out https://example.com now."
    ]

    results = compute_tf_idf_scores(raw_data)

    for res in results:
        print(f"Original: {res['original']}")
        print(f"Tokens: {res['tokens']}")
        # Printing non-zero TF-IDF for brevity
        non_zero_tfidf = {k: round(v, 4) for k, v in res['tf_idf'].items() if v > 0}
        print(f"TF-IDF: {non_zero_tfidf}\n")

# Calling main to demonstrate
main()

Original: Order 3 items, get 1 free!
Tokens: ['order', 'NUMBER', 'items', 'PUNCT', 'get', 'NUMBER', 'free', 'PUNCT']
TF-IDF: {}

Original: Check out https://example.com now.
Tokens: ['check', 'out', 'URL', 'now', 'PUNCT']
TF-IDF: {}



# **Question 2: WordPiece Tokenization**

In [3]:
import re
from collections import defaultdict

# --- Dataset ---
raw_corpus = [
    "The boy hugs the cat.",
    "The boys are hugging the dogs.",
    "The dogs are chasing the cats.",
    "The dog and the cat sit quietly.",
    "The boy is sitting on the dog."
]

# --- 1. Preprocessing (Tokenize by punctuation & lowercase) ---
def preprocess_corpus(corpus):
    word_counts = defaultdict(int)
    for sentence in corpus:
        # Remove trailing punctuation for cleaner words or treat as separate token
        # Simple tokenization: lowercase and split by space, separate end punctuation
        sentence = sentence.lower().strip()
        if sentence.endswith('.'):
            sentence = sentence[:-1] + " ."

        words = sentence.split()
        for word in words:
            word_counts[word] += 1
    return word_counts

# Initialize Vocab: Add characters and '##' prefixed characters
def get_initial_vocab(word_counts):
    vocab = set()
    for word in word_counts:
        vocab.add(word[0])
        for char in word[1:]:
            vocab.add("##" + char)
    return vocab

# Represent words as lists of subword tokens
# e.g., "dogs" -> ["d", "##o", "##g", "##s"]
def split_words(word_counts):
    splits = {}
    for word, count in word_counts.items():
        tokens = [word[0]] + [f"##{c}" for c in word[1:]]
        splits[word] = tokens
    return splits

# --- 2. WordPiece Learning Loop ---
def compute_pair_scores(splits, word_counts):
    # Count frequency of individual tokens and pairs
    token_freqs = defaultdict(int)
    pair_freqs = defaultdict(int)

    for word, freq in word_counts.items():
        tokens = splits[word]
        for i in range(len(tokens)):
            token_freqs[tokens[i]] += freq
            if i < len(tokens) - 1:
                pair = (tokens[i], tokens[i+1])
                pair_freqs[pair] += freq

    # Calculate Score = freq(pair) / (freq(first) * freq(second))
    scores = {}
    for pair, freq in pair_freqs.items():
        score = freq / (token_freqs[pair[0]] * token_freqs[pair[1]])
        scores[pair] = score

    return scores

def merge_pair(a, b, splits):
    new_token = a + b.replace("##", "")
    for word in splits:
        tokens = splits[word]
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        splits[word] = new_tokens
    return new_token

# Main Execution
word_counts = preprocess_corpus(raw_corpus)
vocab = get_initial_vocab(word_counts)
splits = split_words(word_counts)

num_merges = 20
print(f"--- Applying {num_merges} WordPiece Merges ---")

for i in range(num_merges):
    scores = compute_pair_scores(splits, word_counts)
    if not scores:
        break

    best_pair = max(scores, key=scores.get)
    new_token = merge_pair(best_pair[0], best_pair[1], splits)
    vocab.add(new_token)
    print(f"Iter {i+1}: Merged {best_pair} -> '{new_token}'")

# --- 3. Tokenize New Sentence ---
def tokenize_wordpiece(sentence, vocab):
    sentence = sentence.lower().replace(".", " .")
    words = sentence.split()
    encoded_tokens = []

    # Sort vocab by length descending to match longest tokens first
    sorted_vocab = sorted(list(vocab), key=len, reverse=True)

    for word in words:
        # Simple greedy matching from start of word
        # (Real BERT tokenization is slightly more complex, checking all substrings)

        # Try to find the word in vocab directly or break it down
        if word in vocab:
            encoded_tokens.append(word)
            continue

        # Greedy breakdown
        start = 0
        while start < len(word):
            end = len(word)
            matched = False
            while end > start:
                sub = word[start:end]
                # If not start of word, add ## prefix
                if start > 0:
                    sub = "##" + sub

                if sub in vocab:
                    encoded_tokens.append(sub)
                    start = end
                    matched = True
                    break
                end -= 1
            if not matched:
                encoded_tokens.append("[UNK]") # Should not happen if chars are in vocab
                start += 1

    return encoded_tokens

target_sent = "The cat is chasing the dog quietly."
tokens = tokenize_wordpiece(target_sent, vocab)
print(f"\nOriginal: {target_sent}")
print(f"Tokenized: {tokens}")

--- Applying 20 WordPiece Merges ---
Iter 1: Merged ('h', '##u') -> 'hu'
Iter 2: Merged ('q', '##u') -> 'qu'
Iter 3: Merged ('a', '##r') -> 'ar'
Iter 4: Merged ('##l', '##y') -> '##ly'
Iter 5: Merged ('a', '##n') -> 'an'
Iter 6: Merged ('an', '##d') -> 'and'
Iter 7: Merged ('o', '##n') -> 'on'
Iter 8: Merged ('c', '##a') -> 'ca'
Iter 9: Merged ('##i', '##n') -> '##in'
Iter 10: Merged ('s', '##i') -> 'si'
Iter 11: Merged ('qu', '##i') -> 'qui'
Iter 12: Merged ('b', '##o') -> 'bo'
Iter 13: Merged ('bo', '##y') -> 'boy'
Iter 14: Merged ('d', '##o') -> 'do'
Iter 15: Merged ('ca', '##t') -> 'cat'
Iter 16: Merged ('si', '##t') -> 'sit'
Iter 17: Merged ('##t', '##ly') -> '##tly'
Iter 18: Merged ('sit', '##t') -> 'sitt'
Iter 19: Merged ('sitt', '##in') -> 'sittin'
Iter 20: Merged ('##a', '##s') -> '##as'

Original: The cat is chasing the dog quietly.
Tokenized: ['t', '##h', '##e', 'cat', 'i', '##s', 'c', '##h', '##as', '##in', '##g', 't', '##h', '##e', 'do', '##g', 'qui', '##e', '##tly', '.']


# **Question 3: Naive Bayes Classification**

In [5]:
import re
import math
from collections import defaultdict

# --- 1. Dataset ---
data = [
    ("Check out https://example.com for more info!", "Inform"),
    ("Order 3 items, get 1 free! Limited offer!!!", "Promo"),
    ("Your package #12345 will arrive tomorrow.", "Inform"),
    ("Win $1000 now, visit http://winbig.com!!!", "Promo"),
    ("Meeting at 3pm, don't forget to bring the files.", "Reminder"),
    ("Exclusive deal for you: buy 2, get 1 free!!!", "Promo"),
    ("Download the report from https://reports.com.", "Inform"),
    ("The meeting is starting in 10 minutes.", "Reminder"),
    ("Reminder: submit your timesheet by 5pm today.", "Reminder")
]


def preprocess(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', 'URL', text) # Convert URLs
    text = re.sub(r'\d+', 'NUMBER', text)            # Convert Numbers
    text = re.sub(r'[^\w\s]', ' PUNCT ', text)       # Handle Punctuation (keep as token)
    tokens = text.split()
    return tokens

class_counts = defaultdict(int)
feature_counts = defaultdict(lambda: defaultdict(int)) # {class: {feature: count}}
vocab = set()


preprocessed_data = []
for sentence, label in data:
    tokens = preprocess(sentence)
    preprocessed_data.append((tokens, label))

    class_counts[label] += 1


    if len(tokens) > 1:
        for i in range(len(tokens) - 1):
            bigram = (tokens[i], tokens[i+1])
            feature_counts[label][bigram] += 1
            vocab.add(bigram)

total_docs = sum(class_counts.values())
classes = class_counts.keys()
K = 0.3  # Smoothing factor

print("--- Preprocessed Training Sentences ---")
for t, l in preprocessed_data:
    print(f"Label: {l} | Tokens: {t}")

# --- 4. Probability Calculation Helper ---
def get_class_prob(label):
    return math.log(class_counts[label] / total_docs)

def get_feature_prob(feature, label):
    # P(feature | label) = (count(feature, label) + K) / (total_features_in_class + K * |V|)
    count_w_c = feature_counts[label].get(feature, 0)

    # Total count of all bigrams in this class
    count_all_c = sum(feature_counts[label].values())

    prob = (count_w_c + K) / (count_all_c + K * len(vocab))
    return math.log(prob)

# --- 5. Prediction ---
test_sentence = "You will get an exclusive offer in the meeting!"
test_tokens = preprocess(test_sentence)
test_bigrams = [(test_tokens[i], test_tokens[i+1]) for i in range(len(test_tokens)-1)]

print(f"\n--- Test Sentence Processing ---")
print(f"Original: {test_sentence}")
print(f"Tokens: {test_tokens}")
print(f"Bigrams: {test_bigrams}")

print(f"\n--- Prediction Scores ---")
scores = {}

for label in classes:
    # Log Probability = log(P(c)) + sum(log(P(w|c)))
    log_prob = get_class_prob(label)

    feature_sum = 0
    for bigram in test_bigrams:

        p_feat = get_feature_prob(bigram, label)
        feature_sum += p_feat

    total_score = log_prob + feature_sum
    scores[label] = total_score
    print(f"Class: {label:10} | Prior: {log_prob:.4f} | Likelihood Sum: {feature_sum:.4f} | Total: {total_score:.4f}")

predicted_label = max(scores, key=scores.get)
print(f"\n>>> Predicted Label: {predicted_label}")

--- Preprocessed Training Sentences ---
Label: Inform | Tokens: ['check', 'out', 'URL', 'for', 'more', 'info', 'PUNCT']
Label: Promo | Tokens: ['order', 'NUMBER', 'items', 'PUNCT', 'get', 'NUMBER', 'free', 'PUNCT', 'limited', 'offer', 'PUNCT', 'PUNCT', 'PUNCT']
Label: Inform | Tokens: ['your', 'package', 'PUNCT', 'NUMBER', 'will', 'arrive', 'tomorrow', 'PUNCT']
Label: Promo | Tokens: ['win', 'PUNCT', 'NUMBER', 'now', 'PUNCT', 'visit', 'URL']
Label: Reminder | Tokens: ['meeting', 'at', 'NUMBERpm', 'PUNCT', 'don', 'PUNCT', 't', 'forget', 'to', 'bring', 'the', 'files', 'PUNCT']
Label: Promo | Tokens: ['exclusive', 'deal', 'for', 'you', 'PUNCT', 'buy', 'NUMBER', 'PUNCT', 'get', 'NUMBER', 'free', 'PUNCT', 'PUNCT', 'PUNCT']
Label: Inform | Tokens: ['download', 'the', 'report', 'from', 'URL']
Label: Reminder | Tokens: ['the', 'meeting', 'is', 'starting', 'in', 'NUMBER', 'minutes', 'PUNCT']
Label: Reminder | Tokens: ['reminder', 'PUNCT', 'submit', 'your', 'timesheet', 'by', 'NUMBERpm', 'today'